# QuetHoaDon · Đo OCR đọc theo đoạn (gộp khung CRAFT liền nhau)

Bước 1 của kế hoạch cải thiện OCR: thay vì để VietOCR đọc từng khung CRAFT nhỏ (một số bị cắt làm hai, ký tự chỗ chồng đọc
hai lần: `399` + `9,000`, `210, ,000`), gộp các khung liền nhau trên cùng một dòng thành một đoạn rồi đọc một lần
(`src/detection/line_merge.py`, cờ `--merge-lines`).

**Cách chạy:** Settings **GPU T4** (không chọn P100), **Internet On**. Gắn 2 dataset: **Vietnamese Receipts MC_OCR 2021**
và `phase1-result` (hoặc dataset tự tạo từ `phase1_eval.zip`; chỉ cần có `vietocr_ft/` và `tok_ft_n05/`). Bấm **Run All**. Không cần detectron2. Thời gian khoảng 1–1,5 giờ (phần lớn là cell 6).

**So với gì:** bản OCR của cấu hình cuối (`phase1_eval/tok_ft_n05`: VietOCR tinh chỉnh + lọc hộp chồng 0,5). Chỉ chạy
**train và validation**; tập test không được đọc.

**Luật chọn (viết trước khi chạy):** giữ đọc theo đoạn chỉ khi cả ba điều sau đúng

1. **train**: số hóa đơn có tổng các món = tổng chuẩn tăng, và số hóa đơn được sửa nhiều hơn số bị làm hỏng;
2. **validation**: số trường khớp tuyệt đối SELLER + ADDRESS của "trần OCR" không giảm quá 2;
3. thời gian OCR trung bình mỗi ảnh không quá 2 lần bản gốc.

Nếu đạt: bước sau là train lại M1b trên `tok_merge` (notebook khác). Nếu không: ghi kết quả âm, giữ cấu hình cũ.
Dừng ở cell lỗi và gửi output.

In [ ]:
# CELL 1 - Lấy mã nguồn nhánh Thinh/ocr-line-merge và cài thư viện OCR (2-3 phút). Mong đợi: một dòng commit, "xong"
BRANCH = "Thinh/ocr-line-merge"
import os
%cd /kaggle/working
if not os.path.exists('QuetHoaDon'):
    !git clone -q https://github.com/CaoLePhucThinh2203/QuetHoaDon.git
%cd /kaggle/working/QuetHoaDon
!git fetch -q origin && git checkout -q -f -B {BRANCH} origin/{BRANCH} && git log --oneline -1
!pip install -q -r requirements-dev.txt
!pip install -q craft-text-detector vietocr --no-deps
!pip install -q gdown einops prefetch_generator albumentations lmdb 2>&1 | tail -2
print("xong")

In [ ]:
# CELL 2 [KIỂM THỬ] - GPU, dữ liệu, trọng số VietOCR tinh chỉnh, OCR gốc tok_ft_n05 và unit test.
# Mong đợi: "GPU: True Tesla T4", mọi đường dẫn có giá trị, "Ran ... tests" rồi "OK"
import glob, os, sys, json, csv, torch
REPO, W = '/kaggle/working/QuetHoaDon', '/kaggle/working'
sys.path.insert(0, REPO)
print("GPU:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "Không thấy GPU: Settings -> Accelerator -> GPU T4"
one = lambda pattern: sorted(glob.glob(pattern, recursive=True))[-1]
CSV = one('/kaggle/input/**/mcocr_train_df.csv')
IMG_RAW = os.path.dirname(one('/kaggle/input/**/train_images/**/*.jpg'))
VFT1 = one('/kaggle/input/**/vietocr_ft/vietocr_finetuned.pth')
TOKC = os.path.dirname(one('/kaggle/input/**/tok_ft_n05/validation.jsonl'))
for k in ("CSV", "IMG_RAW", "VFT1", "TOKC"):
    print(f"{k} = {globals()[k]}")
os.makedirs(f'{W}/eval', exist_ok=True)
!cd {REPO} && python -m unittest discover -s tests -t . 2>&1 | grep -E "^(Ran|OK|FAILED|ERROR|FAIL)"

In [ ]:
# CELL 3 [KIỂM THỬ] - File chia tập; phải trùng với tok_ft_n05 (cùng hóa đơn train / validation).
# Mong đợi: "train khớp: True (922 ...)" và "validation khớp: True (115 ...)"
from src.training.data import load_jsonl
!python scripts/prepare_dataset.py --csv "{CSV}" --images "{IMG_RAW}" --out {W}/dataset > {W}/dataset_summary.txt
MANIFEST = f'{W}/dataset/mcocr_manifest_split.csv'
split_rows = list(csv.DictReader(open(MANIFEST, encoding='utf-8')))
for sp in ("train", "validation"):
    new = {r['img_id'] for r in split_rows if r['split'] == sp}
    old = {r['img_id'] for r in load_jsonl(os.path.join(TOKC, f'{sp}.jsonl'))}
    print(f"{sp} khớp:", new == old, f"({len(new)} hóa đơn; tok_ft_n05 {len(old)})")

In [ ]:
# CELL 4 [KIỂM THỬ] - Chạy thử 5 ảnh, không gộp và có gộp (1-2 phút).
# Mong đợi: hai dòng "[done] {'ok': 5, ...}", dòng [ocr] đầu có merge_lines=False, dòng sau merge_lines=True
COMMON = f'--images "{IMG_RAW}" --split-manifest {MANIFEST} --weights {VFT1} --nested-threshold 0.5 --device cuda'
!python -u scripts/run_ocr_dataset.py {COMMON} --splits validation --limit 5 --out {W}/smoke_base 2>&1 | grep -E "^\[(ocr|done)\]"
!python -u scripts/run_ocr_dataset.py {COMMON} --splits validation --limit 5 --out {W}/smoke_merge --merge-lines 2>&1 | grep -E "^\[(ocr|done)\]"
for name in ("smoke_base", "smoke_merge"):
    j = json.load(open(sorted(glob.glob(f'{W}/{name}/ocr_annotations/*.json'))[0]))
    print(name, '| số vùng:', j['n_text_regions'], '| giây OCR:', j['timing_sec']['ocr'], '|', [it['text'] for it in j['ocr_items'][:10]])

In [ ]:
# CELL 5 [KIỂM THỬ] - Bản gốc trên 100 ảnh validation đầu (khoảng 5 phút): kiểm tra môi trường tái lập tok_ft_n05 và lấy
# thời gian OCR làm mốc. Mong đợi: "identical words in X/100" với X gần 100 (môi trường giống lần chạy cấu hình cuối)
!python -u scripts/run_ocr_dataset.py {COMMON} --splits validation --limit 100 --out {W}/ocr_base100 2>&1 | grep -E "^\[(ocr|done)\]"
!python scripts/build_token_dataset.py --csv "{CSV}" --ocr-dir {W}/ocr_base100/ocr_annotations --split-manifest {MANIFEST} --out {W}/tok_base100 > /dev/null
!python scripts/compare_ocr_runs.py --reference ft_n05="{TOKC}" --run base100={W}/tok_base100 --out {W}/eval/ocr_base100_vs_ft_n05.json | tail -2

In [ ]:
# CELL 6 - OCR có gộp đoạn trên toàn bộ train + validation (khoảng 50-80 phút). Mong đợi: "[done] {'ok': ...}", failed gần 0
# Nếu phiên bị ngắt: chạy lại cell 1-3 rồi cell này, ảnh đã xong được bỏ qua.
OCR_M = f'{W}/ocr_merge'
!python -u scripts/run_ocr_dataset.py {COMMON} --splits train,validation --out {OCR_M} --merge-lines 2>&1 | grep -E "^\[(ocr|done)\]"

In [ ]:
# CELL 7 [KIỂM THỬ] - Token dataset từ OCR gộp đoạn, rồi "trần OCR" trên validation so với cấu hình cuối (CPU, 1-2 phút).
# Mong đợi: bảng 2 dòng (ft_n05, merge) và dòng "merge vs ft_n05: identical words in ..."
TOKM = f'{W}/tok_merge'
!python scripts/build_token_dataset.py --csv "{CSV}" --ocr-dir {OCR_M}/ocr_annotations --split-manifest {MANIFEST} --out {TOKM} | head -4
!python scripts/compare_ocr_runs.py --reference ft_n05="{TOKC}" --run merge={TOKM} --out {W}/eval/ocr_merge_vs_ft_n05_val.json

In [ ]:
# CELL 8 [KIỂM THỬ] - Mặt hàng (luật hiện tại, chấm bằng tổng chuẩn) trên train và validation cho cả hai bản OCR (CPU, 1 phút).
# Mong đợi: hai dòng, mỗi dòng có số hóa đơn đúng của bản gốc và bản gộp, số sửa được / bị làm hỏng
from src.extraction.line_items import extract_items, check_items
from src.evaluation.metrics import normalize_amount
def item_ok(tok_dir, split):
    out = {}
    for r in load_jsonl(os.path.join(tok_dir, f'{split}.jsonl')):
        gold = {a for a in (normalize_amount(t) for t in r.get('gt', {}).get('TOTAL_COST', [])) if a and a < 1e8}
        if not gold:
            continue
        x = extract_items(r['words'], r['boxes']); c = check_items(x['items'], x['summary_amounts'])
        out[r['img_id']] = (c['sum'] in gold or c['sum_without_discounts'] in gold, c['status'] == 'verified')
    return out
ITEMS = {}
for split in ("train", "validation"):
    a, b = item_ok(TOKC, split), item_ok(TOKM, split)
    common = sorted(set(a) & set(b))
    fixed = sum(b[k][0] and not a[k][0] for k in common); broke = sum(a[k][0] and not b[k][0] for k in common)
    ITEMS[split] = {"n": len(common), "base": sum(a[k][0] for k in common), "merge": sum(b[k][0] for k in common),
                    "base_verified": sum(a[k][1] for k in common), "merge_verified": sum(b[k][1] for k in common),
                    "fixed": fixed, "broken": broke}
    print(f"{split}: tổng các món = tổng chuẩn | gốc {ITEMS[split]['base']}/{len(common)} | gộp {ITEMS[split]['merge']}/{len(common)}"
          f" | sửa {fixed}, hỏng {broke} | verified {ITEMS[split]['base_verified']} -> {ITEMS[split]['merge_verified']}")
for split in ("train", "validation"):
    !python scripts/score_items.py --token-dir {TOKM} --split {split} --out {W}/eval/items_{split}_merge.json > /dev/null
json.dump(ITEMS, open(f'{W}/eval/items_base_vs_merge.json', 'w'), indent=2)

In [ ]:
# CELL 9 [KIỂM THỬ] - Thời gian OCR mỗi ảnh trên cùng 100 ảnh validation (bản gốc ở cell 5, bản gộp ở cell 6).
# Mong đợi: hai dòng giây / ảnh và số vùng trung bình
import statistics
def timing(ocr_dir, ids):
    js = [json.load(open(f'{ocr_dir}/ocr_annotations/{i}.json')) for i in ids]
    return (statistics.mean(j['timing_sec']['ocr'] for j in js), statistics.mean(j['timing_sec']['total'] for j in js),
            statistics.mean(j['n_text_regions'] for j in js))
ids = [os.path.basename(p)[:-5] for p in sorted(glob.glob(f'{W}/ocr_base100/ocr_annotations/*.json'))]
ids = [i for i in ids if os.path.exists(f'{OCR_M}/ocr_annotations/{i}.json')]
TIME = {name: dict(zip(("ocr_sec", "total_sec", "regions"), timing(d, ids))) for name, d in (("base", f'{W}/ocr_base100'), ("merge", OCR_M))}
for name, t in TIME.items():
    print(f"{name}: VietOCR {t['ocr_sec']:.2f} s/ảnh | cả OCR {t['total_sec']:.2f} s/ảnh | {t['regions']:.0f} vùng ({len(ids)} ảnh)")
json.dump(TIME, open(f'{W}/eval/ocr_timing_base_vs_merge.json', 'w'), indent=2)

In [ ]:
# CELL 10 - Áp luật chọn viết sẵn ở đầu notebook và in kết luận.
o = json.load(open(f'{W}/eval/ocr_merge_vs_ft_n05_val.json'))['oracle']
exact = {k: sum(v[f]['exact'] for f in ('SELLER', 'ADDRESS')) for k, v in o.items()}
c1 = ITEMS['train']['merge'] > ITEMS['train']['base'] and ITEMS['train']['fixed'] > ITEMS['train']['broken']
c2 = exact['merge'] >= exact['ft_n05'] - 2
c3 = TIME['merge']['total_sec'] <= 2 * TIME['base']['total_sec']
print("1. train, mặt hàng tốt hơn:", c1, ITEMS['train'])
print("2. validation, SELLER + ADDRESS khớp tuyệt đối:", c2, exact)
print("3. thời gian:", c3, {k: round(v['total_sec'], 2) for k, v in TIME.items()})
print("KẾT LUẬN:", "GIỮ đọc theo đoạn -> train lại M1b trên tok_merge" if (c1 and c2 and c3) else "KHÔNG giữ -> ghi kết quả âm")

In [ ]:
# CELL 11 - Nén kết quả để tải về (tab Output): eval và token dataset gộp đoạn (dùng để train lại M1b nếu giữ)
!cd {W} && zip -q -r line_merge_eval.zip eval tok_merge dataset/mcocr_manifest_split.csv && ls -la line_merge_eval.zip